# 02 – Task 1: Regression on Community Measures

**Question:** Can we predict continuous community-related node properties
using only features that do NOT require knowing the community structure?

## What we predict (13 targets)
| Target | Meaning |
|--------|---------|
| `cada` | Community-aware degree assortativity |
| `edge_contr` | Edge contribution to community |
| `in_mod_deg` | Intra-community degree |
| `participation` | Participation coefficient |
| `beta_star` | Beta-star community overlap measure |
| `l1`, `l2` | L1/L2 distances from community centroid |
| `kl`, `kl2` | KL divergence measures |
| `hd`, `hd2` | Hellinger distance measures |
| `l12`, `l22` | Squared L1/L2 variants |

## Feature sets used
- **Community-free** (8): `lcc`, `bc`, `cc`, `dc`, `ndc`, `ec`, `eccen`, `core`
- **+ node2vec** (16-dim embedding)
- **+ struc2vec** (16-dim embedding)

## Models
Linear Regression, Ridge, Random Forest, XGBoost, LightGBM

## Metrics
R², Spearman ρ, Kendall τ

In [ ]:
# ── CHANGE THIS to switch dataset ──────────────────────────────────────────
DATASET = "facebook"
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.stats import spearmanr, kendalltau
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

ROOT = Path("..")
PATH_COMBINED = ROOT / f"data/04_models/{DATASET}_combined.parquet"

print(f"Dataset: {DATASET}")

## Step 1 – Load combined dataset

Run `01_data_preparation.ipynb` first to generate the combined parquet file.

In [ ]:
df = pd.read_parquet(PATH_COMBINED)

COMMUNITY_FREE   = ["lcc", "bc", "cc", "dc", "ndc", "ec", "eccen", "core"]
REGRESSION_TARGETS = [
    "cada", "edge_contr", "in_mod_deg", "participation",
    "beta_star", "l1", "l2", "kl", "hd", "l12", "l22", "kl2", "hd2",
]
N2V_COLS = [c for c in df.columns if c.startswith("emb_node2vec")]
S2V_COLS = [c for c in df.columns if c.startswith("emb_struc2vec")]

# Feature set: community-free graph features + both embeddings
FEATURES = COMMUNITY_FREE + N2V_COLS + S2V_COLS

print(f"Nodes    : {len(df)}")
print(f"Features : {len(FEATURES)}  "
      f"(8 graph + {len(N2V_COLS)} node2vec + {len(S2V_COLS)} struc2vec)")
print(f"Targets  : {len(REGRESSION_TARGETS)}")

## Step 2 – Train / test split

70% train, 30% test. `random_state=2023` for reproducibility.

In [ ]:
X = df[FEATURES + REGRESSION_TARGETS]   # keep targets alongside features in one df

X_train, X_test = train_test_split(X, test_size=0.3, random_state=2023)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)

print(f"Train: {len(X_train)} nodes | Test: {len(X_test)} nodes")

## Step 3 – Define models

Parameters match those used in the original pipeline (`conf/base/parameters.yml`).

In [ ]:
def make_models():
    """Return dict of {name: unfitted model}."""
    return {
        "LinearRegression": LinearRegression(fit_intercept=True, n_jobs=-1),

        "Ridge": Ridge(alpha=1.0, fit_intercept=True, solver="auto", random_state=2023),

        "RandomForest": RandomForestRegressor(
            n_estimators=1000, max_depth=10,
            min_samples_split=2, min_samples_leaf=1,
            n_jobs=-1, random_state=2023
        ),

        "XGBoost": xgb.XGBRegressor(
            n_estimators=1000, max_depth=10,
            learning_rate=0.1, n_jobs=-1,
            random_state=2023, eval_metric="rmse",
            verbosity=0,
        ),

        "LightGBM": lgb.LGBMRegressor(
            boosting_type="gbdt", objective="regression",
            num_leaves=10, n_jobs=-1,
        ),
    }

print("Models defined:", list(make_models().keys()))

## Step 4 – Train and evaluate

For each **target** × **model** combination we compute:
- **R²** – coefficient of determination (1 = perfect, 0 = baseline mean)
- **Spearman ρ** – rank correlation (robust to outliers)
- **Kendall τ** – another rank correlation, more conservative

In [ ]:
records = []

for target in REGRESSION_TARGETS:
    y_train = X_train[target]
    y_test  = X_test[target]
    X_tr    = X_train[FEATURES]
    X_te    = X_test[FEATURES]

    for model_name, model in make_models().items():
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)

        records.append({
            "target"    : target,
            "model"     : model_name,
            "R2"        : round(r2_score(y_test, y_pred), 4),
            "Spearman"  : round(spearmanr(y_test, y_pred).statistic, 4),
            "Kendall"   : round(kendalltau(y_test, y_pred).statistic, 4),
        })

        print(f"  {target:15s} | {model_name:16s} | "
              f"R²={records[-1]['R2']:.3f}  "
              f"ρ={records[-1]['Spearman']:.3f}  "
              f"τ={records[-1]['Kendall']:.3f}")

results = pd.DataFrame(records)
print(f"\nDone. {len(results)} rows in results table.")

## Step 5 – Results table

In [ ]:
# Pivot: rows = target, columns = model × metric
pivot = results.pivot_table(
    index="target", columns="model", values=["R2", "Spearman", "Kendall"]
)
pivot.columns = [f"{metric}_{model}" for metric, model in pivot.columns]
pivot = pivot.round(4)
pivot

In [ ]:
# Best model per target (by Spearman)
best = (
    results.sort_values("Spearman", ascending=False)
           .groupby("target")
           .first()[["model", "R2", "Spearman", "Kendall"]]
)
print("Best model per target (by Spearman ρ):")
best

In [ ]:
# Save results
PATH_OUT = ROOT / f"data/05_reporting/{DATASET}_task1.parquet"
results.to_parquet(PATH_OUT, index=False)
print(f"Saved to {PATH_OUT}")